# Ensemble

Este notebook mantém o baseline de Leal et al. e executa o treinamento em:

1. `data/original/`
2. `data/pair_controlled/seed_*`
3. `data/max_cross_split/seed_*`

A seed do modelo permanece fixa em 40.

Para cada modelo treinado são salvos:

- métricas gerais;
- TP, TN, FP e FN;
- matriz de confusão;
- análise dos sinais de trocadilho presentes nos falsos negativos;
- gráfico da análise de erros no mesmo formato conceitual usado no artigo.

Nenhum exemplo textual de erro é salvo.


In [44]:
from pathlib import Path
import json
import re
import sys
import platform

import joblib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd

from sklearn import __version__ as sklearn_version
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

In [ ]:
MODEL_SEED = 40

SPLIT_SEEDS = [
    13,
    21,
    40,
    42,
    73,
]

STRATEGIES = [
    "pair_controlled",
    "max_cross_split",
]

SPLIT_NAMES = [
    "train",
    "validation",
    "test",
]

EXPECTED_SPLIT_COUNTS = {
    "train": 3990,
    "validation": 570,
    "test": 1140,
}

EXPECTED_CLASS_COUNTS = {
    "train": {0: 1995, 1: 1995},
    "validation": {0: 285, 1: 285},
    "test": {0: 570, 1: 570},
}

ERROR_CATEGORIES = [
    "none",
    "homophone_only",
    "homograph_only",
    "both",
]

In [46]:
def find_project_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()

    while True:
        if (current / "data" / "original").is_dir():
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        "Could not locate the project root containing data/original/."
    )

In [47]:
PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
ORIGINAL_DIR = DATA_DIR / "original"
PUNS_PATH = DATA_DIR / "puns.json"
RESULTS_DIR = PROJECT_ROOT / "results" / "ensemble"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if not PUNS_PATH.is_file():
    raise FileNotFoundError(
        f"Missing file: {PUNS_PATH}"
    )

print("Project root:", PROJECT_ROOT)
print("Original corpus:", ORIGINAL_DIR)
print("Puns:", PUNS_PATH)
print("Results:", RESULTS_DIR)

Project root: /home/avelar/pun-detection-split-analysis
Original corpus: /home/avelar/pun-detection-split-analysis/data/original
Puns: /home/avelar/pun-detection-split-analysis/data/puns.json
Results: /home/avelar/pun-detection-split-analysis/results/ensemble


In [48]:
nltk.download(
    "stopwords",
    quiet=True,
)

stop_words = nltk.corpus.stopwords.words(
    "portuguese"
)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn_version)
print("Model seed:", MODEL_SEED)
print("Split seeds:", SPLIT_SEEDS)

Python: 3.14.4
Platform: Linux-7.0.0-28-generic-x86_64-with-glibc2.43
NumPy: 2.5.2
Pandas: 3.0.5
scikit-learn: 1.9.0
Model seed: 40
Split seeds: [13, 21, 40, 42, 73]


/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/avelar/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


## Carregamento e validação


In [49]:
def load_data(file_path):
    return pd.read_json(
        file_path,
        lines=True,
    )

In [50]:
def create_x_y(dataframe):
    return (
        dataframe["text"],
        dataframe["label"],
    )

In [51]:
def load_split_directory(split_dir):
    split_dir = Path(split_dir)

    paths = {
        split_name: (
            split_dir
            / f"{split_name}.jsonl"
        )
        for split_name in SPLIT_NAMES
    }

    for split_name, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing {split_name} file: {path}"
            )

    return {
        split_name: load_data(path)
        for split_name, path in paths.items()
    }

In [52]:
def validate_input_splits(
    split_data,
    run_name,
):
    all_ids = []

    for split_name in SPLIT_NAMES:
        split_df = split_data[split_name]

        expected_size = EXPECTED_SPLIT_COUNTS[
            split_name
        ]

        expected_classes = EXPECTED_CLASS_COUNTS[
            split_name
        ]

        if len(split_df) != expected_size:
            raise ValueError(
                f"{run_name}/{split_name}: expected "
                f"{expected_size} examples, found {len(split_df)}."
            )

        required_columns = {
            "id",
            "text",
            "label",
        }

        missing_columns = (
            required_columns
            - set(split_df.columns)
        )

        if missing_columns:
            raise ValueError(
                f"{run_name}/{split_name}: missing columns "
                f"{sorted(missing_columns)}."
            )

        split_df["label"] = (
            split_df["label"]
            .astype(int)
        )

        observed_classes = (
            split_df["label"]
            .value_counts()
            .sort_index()
            .to_dict()
        )

        if observed_classes != expected_classes:
            raise ValueError(
                f"{run_name}/{split_name}: expected class distribution "
                f"{expected_classes}, found {observed_classes}."
            )

        if split_df["id"].duplicated().any():
            raise ValueError(
                f"{run_name}/{split_name}: duplicated IDs."
            )

        all_ids.extend(
            split_df["id"]
            .astype(str)
            .tolist()
        )

    if len(all_ids) != 5700:
        raise ValueError(
            f"{run_name}: expected 5700 total IDs."
        )

    if len(set(all_ids)) != 5700:
        raise ValueError(
            f"{run_name}: IDs overlap across splits."
        )

In [53]:
def build_leal_baseline():
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        stop_words=stop_words,
    )

    rf_model = RandomForestClassifier(
        n_estimators=100,
        criterion="entropy",
        max_depth=15,
        random_state=40,
    )

    lr_model = LogisticRegression(
        random_state=40,
        max_iter=100,
    )

    svm_model = SVC(
        probability=True,
        random_state=40,
    )

    voting_model = VotingClassifier(
        estimators=[
            ("rf", rf_model),
            ("lr", lr_model),
            ("svm", svm_model),
        ],
        voting="soft",
        n_jobs=30,
    )

    return (
        vectorizer,
        voting_model,
    )

In [54]:
with PUNS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    puns_data = json.load(file)

if not isinstance(
    puns_data,
    list,
):
    raise ValueError(
        "data/puns.json must contain a JSON list."
    )

puns_by_id = {}

for item in puns_data:
    pun_id = str(
        item["id"]
    )

    if pun_id in puns_by_id:
        raise ValueError(
            f"Duplicated pun annotation ID: {pun_id}"
        )

    puns_by_id[
        pun_id
    ] = item

print(
    "Annotated puns:",
    len(puns_by_id),
)

Annotated puns: 2850


In [55]:
def corpus_id_to_pun_id(
    example_id,
):
    return re.sub(
        r"\.[HN]$",
        "",
        str(example_id),
    )


def classify_punning_sign(sign):
    homograph = bool(
        sign.get(
            "homograph",
            False,
        )
    )

    homophone = bool(
        sign.get(
            "homophone",
            False,
        )
    )

    if homograph and homophone:
        return "both"

    if homograph:
        return "homograph_only"

    if homophone:
        return "homophone_only"

    return "none"

In [56]:
def evaluate_predictions(
    y_true,
    y_pred,
):
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=[
            "non_pun",
            "pun",
        ],
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "precision_weighted": float(
            precision_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "recall_weighted": float(
            recall_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "f1_weighted": float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }

    return (
        metrics,
        report_dict,
        cm,
    )

In [57]:
def count_false_negative_punning_signs(
    test_df,
    y_pred,
):
    counts = {
        category: 0
        for category in ERROR_CATEGORIES
    }

    false_negative_count = 0
    annotated_false_negative_count = 0
    missing_annotation_count = 0

    test_ids = (
        test_df["id"]
        .astype(str)
        .tolist()
    )

    y_true = (
        test_df["label"]
        .astype(int)
        .to_numpy()
    )

    y_pred = np.asarray(
        y_pred
    ).astype(int)

    for example_id, true_label, predicted_label in zip(
        test_ids,
        y_true,
        y_pred,
    ):
        if not (
            true_label == 1
            and predicted_label == 0
        ):
            continue

        false_negative_count += 1

        pun_id = corpus_id_to_pun_id(
            example_id
        )

        annotation = puns_by_id.get(
            pun_id
        )

        if annotation is None:
            missing_annotation_count += 1
            continue

        annotated_false_negative_count += 1

        for sign in annotation.get(
            "signs",
            [],
        ):
            category = classify_punning_sign(
                sign
            )

            counts[
                category
            ] += 1

    return {
        "fn_instances": int(
            false_negative_count
        ),
        "annotated_fn_instances": int(
            annotated_false_negative_count
        ),
        "missing_fn_annotations": int(
            missing_annotation_count
        ),
        "none": int(
            counts["none"]
        ),
        "homophone_only": int(
            counts["homophone_only"]
        ),
        "homograph_only": int(
            counts["homograph_only"]
        ),
        "both": int(
            counts["both"]
        ),
        "total_punning_signs_in_fn": int(
            sum(
                counts.values()
            )
        ),
    }

In [58]:
def save_error_analysis_plot(
    error_analysis,
    output_path,
    title,
):
    labels = [
        "None",
        "Homophone only",
        "Homograph only",
        "Both",
    ]

    values = [
        error_analysis[
            "none"
        ],
        error_analysis[
            "homophone_only"
        ],
        error_analysis[
            "homograph_only"
        ],
        error_analysis[
            "both"
        ],
    ]

    fig, ax = plt.subplots(
        figsize=(7, 4.5)
    )

    bars = ax.bar(
        labels,
        values,
    )

    ax.set_ylabel(
        "Number of punning signs in false negatives"
    )

    ax.set_title(
        title
    )

    ax.tick_params(
        axis="x",
        rotation=15,
    )

    for bar, value in zip(
        bars,
        values,
    ):
        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            str(value),
            ha="center",
            va="bottom",
        )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

In [59]:
def save_run_outputs(
    output_dir,
    vectorizer,
    voting_model,
    metrics,
    report_dict,
    cm,
    error_analysis,
    metadata,
):
    output_dir = Path(
        output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    confusion_counts_df = pd.DataFrame(
        [
            {
                "condition": metadata[
                    "condition"
                ],
                "split_seed": metadata[
                    "split_seed"
                ],
                "tp": metrics[
                    "tp"
                ],
                "tn": metrics[
                    "tn"
                ],
                "fp": metrics[
                    "fp"
                ],
                "fn": metrics[
                    "fn"
                ],
            }
        ]
    )

    confusion_counts_df.to_csv(
        output_dir
        / "confusion_counts.csv",
        index=False,
        encoding="utf-8",
    )

    pd.DataFrame(
        cm,
        index=[
            "true_non_pun",
            "true_pun",
        ],
        columns=[
            "pred_non_pun",
            "pred_pun",
        ],
    ).to_csv(
        output_dir
        / "confusion_matrix.csv",
        encoding="utf-8",
    )

    pd.DataFrame(
        [
            {
                "condition": metadata[
                    "condition"
                ],
                "split_seed": metadata[
                    "split_seed"
                ],
                **error_analysis,
            }
        ]
    ).to_csv(
        output_dir
        / "error_analysis.csv",
        index=False,
        encoding="utf-8",
    )

    pd.DataFrame(
        report_dict
    ).T.to_csv(
        output_dir
        / "classification_report.csv",
        encoding="utf-8",
    )

    with (
        output_dir
        / "metrics.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metrics,
            file,
            ensure_ascii=False,
            indent=2,
        )

    with (
        output_dir
        / "metadata.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            ensure_ascii=False,
            indent=2,
        )

    save_error_analysis_plot(
        error_analysis=error_analysis,
        output_path=(
            output_dir
            / "error_analysis.png"
        ),
        title=(
            metadata["condition"]
            if metadata["split_seed"] is None
            else (
                f"{metadata['condition']} "
                f"- seed {metadata['split_seed']}"
            )
        ),
    )

    joblib.dump(
        {
            "vectorizer": vectorizer,
            "model": voting_model,
        },
        output_dir
        / "ensemble.joblib",
    )

In [60]:
def run_ensemble(
    split_dir,
    condition,
    split_seed,
    output_dir,
):
    split_data = load_split_directory(
        split_dir
    )

    validate_input_splits(
        split_data,
        condition,
    )

    train_df = split_data[
        "train"
    ]

    validation_df = split_data[
        "validation"
    ]

    test_df = split_data[
        "test"
    ]

    x_train, y_train = create_x_y(
        train_df
    )

    x_validation, y_validation = create_x_y(
        validation_df
    )

    x_test, y_test = create_x_y(
        test_df
    )

    vectorizer, voting_model = (
        build_leal_baseline()
    )

    x_train_vectorized = (
        vectorizer.fit_transform(
            x_train
        )
    )

    x_validation_vectorized = (
        vectorizer.transform(
            x_validation
        )
    )

    x_test_vectorized = (
        vectorizer.transform(
            x_test
        )
    )

    _ = (
        x_validation_vectorized,
        y_validation,
    )

    voting_model.fit(
        x_train_vectorized,
        y_train,
    )

    y_test_pred = (
        voting_model.predict(
            x_test_vectorized
        )
    )

    (
        metrics,
        report_dict,
        cm,
    ) = evaluate_predictions(
        y_test,
        y_test_pred,
    )

    error_analysis = (
        count_false_negative_punning_signs(
            test_df=test_df,
            y_pred=y_test_pred,
        )
    )

    metadata = {
        "model": "ensemble_leal_baseline",
        "condition": condition,
        "split_seed": split_seed,
        "model_seed": MODEL_SEED,
        "input_directory": str(
            Path(
                split_dir
            ).resolve()
        ),
        "train_examples": len(
            train_df
        ),
        "validation_examples": len(
            validation_df
        ),
        "test_examples": len(
            test_df
        ),
        "tfidf_ngram_range": [
            1,
            2,
        ],
        "portuguese_stopwords": True,
        "random_forest": {
            "n_estimators": 100,
            "criterion": "entropy",
            "max_depth": 15,
            "random_state": 40,
        },
        "logistic_regression": {
            "random_state": 40,
            "max_iter": 100,
        },
        "svm": {
            "probability": True,
            "random_state": 40,
        },
        "voting": "soft",
        "n_jobs": 30,
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn_version,
    }

    save_run_outputs(
        output_dir=output_dir,
        vectorizer=vectorizer,
        voting_model=voting_model,
        metrics=metrics,
        report_dict=report_dict,
        cm=cm,
        error_analysis=error_analysis,
        metadata=metadata,
    )

    print("=" * 80)
    print("Condition:", condition)
    print("Split seed:", split_seed)
    print(
        classification_report(
            y_test,
            y_test_pred,
            labels=[0, 1],
            target_names=[
                "non_pun",
                "pun",
            ],
            digits=4,
            zero_division=0,
        )
    )
    print(
        "TP:",
        metrics["tp"],
    )
    print(
        "TN:",
        metrics["tn"],
    )
    print(
        "FP:",
        metrics["fp"],
    )
    print(
        "FN:",
        metrics["fn"],
    )
    print(
        "Error analysis:",
        {
            category: error_analysis[
                category
            ]
            for category in ERROR_CATEGORIES
        },
    )

    return {
        "condition": condition,
        "split_seed": split_seed,
        "model_seed": MODEL_SEED,
        **metrics,
        **error_analysis,
    }

In [61]:
original_result = run_ensemble(
    split_dir=ORIGINAL_DIR,
    condition="original",
    split_seed=None,
    output_dir=(
        RESULTS_DIR
        / "original"
    ),
)

original_result_df = pd.DataFrame(
    [
        original_result
    ]
)

display(
    original_result_df
)

original_result_df.to_csv(
    RESULTS_DIR
    / "original_result.csv",
    index=False,
    encoding="utf-8",
)

/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Condition: original
Split seed: None
              precision    recall  f1-score   support

     non_pun     0.7927    0.8053    0.7990       570
         pun     0.8021    0.7895    0.7958       570

    accuracy                         0.7974      1140
   macro avg     0.7974    0.7974    0.7974      1140
weighted avg     0.7974    0.7974    0.7974      1140

TP: 450
TN: 459
FP: 111
FN: 120
Error analysis: {'none': 71, 'homophone_only': 24, 'homograph_only': 1, 'both': 26}


,condition,split_seed,model_seed,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,...,fp,fn,fn_instances,annotated_fn_instances,missing_fn_annotations,none,homophone_only,homograph_only,both,total_punning_signs_in_fn
0,original,None,40,0.797368,0.797443,0.797368,0.797356,0.797443,0.797368,0.797356,...,111,120,120,120,0,71,24,1,26,122


In [62]:
pair_controlled_results = []

for split_seed in SPLIT_SEEDS:
    result = run_ensemble(
        split_dir=(
            DATA_DIR
            / "pair_controlled"
            / f"seed_{split_seed}"
        ),
        condition="pair_controlled",
        split_seed=split_seed,
        output_dir=(
            RESULTS_DIR
            / "pair_controlled"
            / f"seed_{split_seed}"
        ),
    )

    pair_controlled_results.append(
        result
    )

pair_controlled_results_df = pd.DataFrame(
    pair_controlled_results
)

display(
    pair_controlled_results_df
)

pair_controlled_results_df.to_csv(
    RESULTS_DIR
    / "pair_controlled"
    / "runs.csv",
    index=False,
    encoding="utf-8",
)

/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Condition: pair_controlled
Split seed: 13
              precision    recall  f1-score   support

     non_pun     0.4778    0.7000    0.5680       570
         pun     0.4393    0.2351    0.3063       570

    accuracy                         0.4675      1140
   macro avg     0.4586    0.4675    0.4371      1140
weighted avg     0.4586    0.4675    0.4371      1140

TP: 134
TN: 399
FP: 171
FN: 436
Error analysis: {'none': 276, 'homophone_only': 89, 'homograph_only': 1, 'both': 77}


/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Condition: pair_controlled
Split seed: 21
              precision    recall  f1-score   support

     non_pun     0.4718    0.5579    0.5113       570
         pun     0.4592    0.3754    0.4131       570

    accuracy                         0.4667      1140
   macro avg     0.4655    0.4667    0.4622      1140
weighted avg     0.4655    0.4667    0.4622      1140

TP: 214
TN: 318
FP: 252
FN: 356
Error analysis: {'none': 230, 'homophone_only': 67, 'homograph_only': 1, 'both': 66}


/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Condition: pair_controlled
Split seed: 42
              precision    recall  f1-score   support

     non_pun     0.4808    0.7246    0.5780       570
         pun     0.4413    0.2175    0.2914       570

    accuracy                         0.4711      1140
   macro avg     0.4610    0.4711    0.4347      1140
weighted avg     0.4610    0.4711    0.4347      1140

TP: 124
TN: 413
FP: 157
FN: 446
Error analysis: {'none': 291, 'homophone_only': 90, 'homograph_only': 1, 'both': 75}


/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Condition: pair_controlled
Split seed: 73
              precision    recall  f1-score   support

     non_pun     0.4764    0.7088    0.5698       570
         pun     0.4315    0.2211    0.2923       570

    accuracy                         0.4649      1140
   macro avg     0.4540    0.4649    0.4311      1140
weighted avg     0.4540    0.4649    0.4311      1140

TP: 126
TN: 404
FP: 166
FN: 444
Error analysis: {'none': 288, 'homophone_only': 92, 'homograph_only': 0, 'both': 76}


,condition,split_seed,model_seed,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,...,fp,fn,fn_instances,annotated_fn_instances,missing_fn_annotations,none,homophone_only,homograph_only,both,total_punning_signs_in_fn
0,pair_controlled,13,40,0.467544,0.458594,0.467544,0.437129,0.458594,0.467544,0.437129,...,171,436,436,436,0,276,89,1,77,443
1,pair_controlled,21,40,0.466667,0.465519,0.466667,0.462191,0.465519,0.466667,0.462191,...,252,356,356,356,0,230,67,1,66,364
2,pair_controlled,40,40,0.464912,0.459641,0.464912,0.446850,0.459641,0.464912,0.446850,...,202,408,408,408,0,253,81,1,83,418
3,pair_controlled,42,40,0.471053,0.461036,0.471053,0.434724,0.461036,0.471053,0.434724,...,157,446,446,446,0,291,90,1,75,457
4,pair_controlled,73,40,0.464912,0.453961,0.464912,0.431080,0.453961,0.464912,0.431080,...,166,444,444,444,0,288,92,0,76,456


In [64]:
max_cross_split_results = []

for split_seed in SPLIT_SEEDS:
    result = run_ensemble(
        split_dir=(
            DATA_DIR
            / "max_cross_split"
            / f"seed_{split_seed}"
        ),
        condition="max_cross_split",
        split_seed=split_seed,
        output_dir=(
            RESULTS_DIR
            / "max_cross_split"
            / f"seed_{split_seed}"
        ),
    )

    max_cross_split_results.append(
        result
    )

max_cross_split_results_df = pd.DataFrame(
    max_cross_split_results
)

display(
    max_cross_split_results_df
)

max_cross_split_results_df.to_csv(
    RESULTS_DIR
    / "max_cross_split"
    / "runs.csv",
    index=False,
    encoding="utf-8",
)

/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Condition: max_cross_split
Split seed: 13
              precision    recall  f1-score   support

     non_pun     0.9615    0.9632    0.9623       570
         pun     0.9631    0.9614    0.9622       570

    accuracy                         0.9623      1140
   macro avg     0.9623    0.9623    0.9623      1140
weighted avg     0.9623    0.9623    0.9623      1140

TP: 548
TN: 549
FP: 21
FN: 22
Error analysis: {'none': 17, 'homophone_only': 4, 'homograph_only': 0, 'both': 3}


/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Condition: max_cross_split
Split seed: 21
              precision    recall  f1-score   support

     non_pun     0.9784    0.9526    0.9653       570
         pun     0.9538    0.9789    0.9662       570

    accuracy                         0.9658      1140
   macro avg     0.9661    0.9658    0.9658      1140
weighted avg     0.9661    0.9658    0.9658      1140

TP: 558
TN: 543
FP: 27
FN: 12
Error analysis: {'none': 6, 'homophone_only': 3, 'homograph_only': 0, 'both': 5}


/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Condition: max_cross_split
Split seed: 42
              precision    recall  f1-score   support

     non_pun     0.9613    0.9579    0.9596       570
         pun     0.9580    0.9614    0.9597       570

    accuracy                         0.9596      1140
   macro avg     0.9597    0.9596    0.9596      1140
weighted avg     0.9597    0.9596    0.9596      1140

TP: 548
TN: 546
FP: 24
FN: 22
Error analysis: {'none': 12, 'homophone_only': 7, 'homograph_only': 0, 'both': 4}


/home/avelar/pun-detection-split-analysis/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Condition: max_cross_split
Split seed: 73
              precision    recall  f1-score   support

     non_pun     0.9661    0.9509    0.9584       570
         pun     0.9516    0.9667    0.9591       570

    accuracy                         0.9588      1140
   macro avg     0.9589    0.9588    0.9588      1140
weighted avg     0.9589    0.9588    0.9588      1140

TP: 551
TN: 542
FP: 28
FN: 19
Error analysis: {'none': 8, 'homophone_only': 3, 'homograph_only': 0, 'both': 8}


,condition,split_seed,model_seed,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,...,fp,fn,fn_instances,annotated_fn_instances,missing_fn_annotations,none,homophone_only,homograph_only,both,total_punning_signs_in_fn
0,max_cross_split,13,40,0.962281,0.962282,0.962281,0.962281,0.962282,0.962281,0.962281,...,21,22,22,22,0,17,4,0,3,24
1,max_cross_split,21,40,0.965789,0.966112,0.965789,0.965784,0.966112,0.965789,0.965784,...,27,12,12,12,0,6,3,0,5,14
2,max_cross_split,40,40,0.957018,0.957053,0.957018,0.957017,0.957053,0.957018,0.957017,...,27,22,22,22,0,22,2,0,2,26
3,max_cross_split,42,40,0.959649,0.959655,0.959649,0.959649,0.959655,0.959649,0.959649,...,24,22,22,22,0,12,7,0,4,23
4,max_cross_split,73,40,0.958772,0.958886,0.958772,0.958769,0.958886,0.958772,0.958769,...,28,19,19,19,0,8,3,0,8,19


In [ ]:
review_results_df = pd.concat(
    [
        pair_controlled_results_df,
        max_cross_split_results_df,
    ],
    ignore_index=True,
)

all_results_df = pd.concat(
    [
        original_result_df,
        review_results_df,
    ],
    ignore_index=True,
)

display(
    all_results_df
)

all_results_df.to_csv(
    RESULTS_DIR
    / "all_runs.csv",
    index=False,
    encoding="utf-8",
)

,condition,split_seed,model_seed,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,...,fp,fn,fn_instances,annotated_fn_instances,missing_fn_annotations,none,homophone_only,homograph_only,both,total_punning_signs_in_fn
0,original,None,40,0.797368,0.797443,0.797368,0.797356,0.797443,0.797368,0.797356,...,111,120,120,120,0,71,24,1,26,122
1,pair_controlled,13,40,0.467544,0.458594,0.467544,0.437129,0.458594,0.467544,0.437129,...,171,436,436,436,0,276,89,1,77,443
2,pair_controlled,21,40,0.466667,0.465519,0.466667,0.462191,0.465519,0.466667,0.462191,...,252,356,356,356,0,230,67,1,66,364
3,pair_controlled,40,40,0.464912,0.459641,0.464912,0.446850,0.459641,0.464912,0.446850,...,202,408,408,408,0,253,81,1,83,418
4,pair_controlled,42,40,0.471053,0.461036,0.471053,0.434724,0.461036,0.471053,0.434724,...,157,446,446,446,0,291,90,1,75,457
5,pair_controlled,73,40,0.464912,0.453961,0.464912,0.431080,0.453961,0.464912,0.431080,...,166,444,444,444,0,288,92,0,76,456
6,random_instance,13,40,0.809649,0.810959,0.809649,0.809448,0.810959,0.809649,0.809448,...,90,127,127,127,0,77,30,0,23,130
7,random_instance,21,40,0.792982,0.795254,0.792982,0.792583,0.795254,0.792982,0.792583,...,93,143,143,143,0,85,31,0,30,146
8,random_instance,40,40,0.810526,0.810542,0.810526,0.810524,0.810542,0.810526,0.810524,...,106,110,110,110,0,72,18,1,22,113
9,random_instance,42,40,0.807018,0.809594,0.807018,0.806615,0.809594,0.807018,0.806615,...,84,136,136,136,0,86,30,0,24,140


In [66]:
confusion_counts_all_df = all_results_df[
    [
        "condition",
        "split_seed",
        "model_seed",
        "tp",
        "tn",
        "fp",
        "fn",
    ]
].copy()

display(
    confusion_counts_all_df
)

confusion_counts_all_df.to_csv(
    RESULTS_DIR
    / "confusion_counts_all.csv",
    index=False,
    encoding="utf-8",
)

,condition,split_seed,model_seed,tp,tn,fp,fn
0,original,None,40,450,459,111,120
1,pair_controlled,13,40,134,399,171,436
2,pair_controlled,21,40,214,318,252,356
3,pair_controlled,40,40,162,368,202,408
4,pair_controlled,42,40,124,413,157,446
5,pair_controlled,73,40,126,404,166,444
6,random_instance,13,40,443,480,90,127
7,random_instance,21,40,427,477,93,143
8,random_instance,40,40,460,464,106,110
9,random_instance,42,40,434,486,84,136


In [67]:
error_analysis_all_df = all_results_df[
    [
        "condition",
        "split_seed",
        "model_seed",
        "fn_instances",
        "annotated_fn_instances",
        "missing_fn_annotations",
        "none",
        "homophone_only",
        "homograph_only",
        "both",
        "total_punning_signs_in_fn",
    ]
].copy()

display(
    error_analysis_all_df
)

error_analysis_all_df.to_csv(
    RESULTS_DIR
    / "error_analysis_all.csv",
    index=False,
    encoding="utf-8",
)

,condition,split_seed,model_seed,fn_instances,annotated_fn_instances,missing_fn_annotations,none,homophone_only,homograph_only,both,total_punning_signs_in_fn
0,original,None,40,120,120,0,71,24,1,26,122
1,pair_controlled,13,40,436,436,0,276,89,1,77,443
2,pair_controlled,21,40,356,356,0,230,67,1,66,364
3,pair_controlled,40,40,408,408,0,253,81,1,83,418
4,pair_controlled,42,40,446,446,0,291,90,1,75,457
5,pair_controlled,73,40,444,444,0,288,92,0,76,456
6,random_instance,13,40,127,127,0,77,30,0,23,130
7,random_instance,21,40,143,143,0,85,31,0,30,146
8,random_instance,40,40,110,110,0,72,18,1,22,113
9,random_instance,42,40,136,136,0,86,30,0,24,140


In [68]:
SUMMARY_METRICS = [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "tp",
    "tn",
    "fp",
    "fn",
    "none",
    "homophone_only",
    "homograph_only",
    "both",
]

summary_rows = []

for strategy in STRATEGIES:
    strategy_df = (
        review_results_df.loc[
            review_results_df[
                "condition"
            ]
            == strategy
        ]
    )

    summary_row = {
        "condition": strategy,
        "runs": len(
            strategy_df
        ),
    }

    for metric in SUMMARY_METRICS:
        summary_row[
            f"{metric}_mean"
        ] = float(
            strategy_df[
                metric
            ].mean()
        )

        summary_row[
            f"{metric}_std"
        ] = float(
            strategy_df[
                metric
            ].std(
                ddof=1
            )
        )

    summary_rows.append(
        summary_row
    )

summary_df = pd.DataFrame(
    summary_rows
)

display(
    summary_df
)

summary_df.to_csv(
    RESULTS_DIR
    / "summary_mean_std.csv",
    index=False,
    encoding="utf-8",
)

,condition,runs,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,...,fn_mean,fn_std,none_mean,none_std,homophone_only_mean,homophone_only_std,homograph_only_mean,homograph_only_std,both_mean,both_std
0,pair_controlled,5,0.467018,0.002527,0.459750,0.004177,0.467018,0.002527,0.442395,0.012513,...,418.0,37.841776,267.6,25.793410,83.8,10.281051,0.8,0.447214,75.4,6.107373
1,random_instance,5,0.806491,0.007787,0.807738,0.007048,0.806491,0.007787,0.806288,0.007932,...,125.4,14.741099,77.4,8.203658,26.2,5.848077,0.2,0.447214,24.4,3.209361
2,max_cross_split,5,0.960702,0.003420,0.960798,0.003515,0.960702,0.003420,0.960700,0.003418,...,19.4,4.335897,13.0,6.557439,3.8,1.923538,0.0,0.000000,4.4,2.302173
